# Phân loại nhị phân phi tuyến bằng mạng MLP sử dụng PyTorch

Notebook này hướng dẫn bạn cách tự định nghĩa một **Custom Dataset** từ dữ liệu NumPy/Pandas, xây dựng mạng **Multi-Layer Perceptron (MLP)** để phân loại dữ liệu phi tuyến dạng hai nửa mặt trăng (Moons dataset), và trực quan hóa **Ranh giới quyết định (Decision Boundary)** bằng **PyTorch** và **Matplotlib**.

### Nội dung chính:
1. Cách tạo một **Custom Dataset** bằng cách kế thừa lớp `torch.utils.data.Dataset`.
2. Cách sử dụng `DataLoader` để chia batch dữ liệu.
3. Xây dựng mạng MLP phân loại nhị phân sử dụng hàm kích hoạt **ReLU** cho các tầng ẩn và **Sigmoid** ở tầng ra.
4. Sử dụng hàm mất mát **Binary Cross-Entropy (BCE)**.
5. Viết vòng lặp huấn luyện (Training Loop) và vẽ ranh giới quyết định để thấy trực quan cách mạng nơ-ron học phân tách dữ liệu.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt

# Thiết lập random seed
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Thiết bị đang sử dụng: {device}")

---
## PHẦN 1: TỰ ĐỊNH NGHĨA CUSTOM DATASET

Trong thực tế, dữ liệu của bạn thường được load từ file CSV (qua Pandas) hoặc ảnh từ thư mục cục bộ. Để nạp dữ liệu này vào PyTorch, chúng ta cần tạo một lớp kế thừa từ `torch.utils.data.Dataset` và override 3 phương thức bắt buộc:
1. **`__init__(self, ...)`**: Khởi tạo lớp, lưu trữ dữ liệu (X, y) hoặc đường dẫn file.
2. **`__len__(self)`**: Trả về tổng số lượng mẫu dữ liệu (ví dụ: `return len(self.X)`).
3. **`__getitem__(self, idx)`**: Trả về 1 mẫu dữ liệu cụ thể kèm nhãn tương ứng ở vị trí chỉ số `idx`. Tại đây ta phải chuyển đổi dữ liệu sang dạng **PyTorch Tensor**.

In [ ]:
# 1. Sinh dữ liệu phi tuyến bằng scikit-learn
X_raw, y_raw = make_moons(n_samples=500, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)

# Chuẩn hóa dữ liệu về mean=0, std=1 để tối ưu hóa hội tụ
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 2. Định nghĩa Custom Dataset kế thừa từ torch.utils.data.Dataset
class MoonsDataset(Dataset):
    def __init__(self, X, y):
        # Chuyển đổi mảng NumPy sang PyTorch Tensors
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1) # Chuyển nhãn thành shape (N, 1)
        
    def __len__(self):
        return len(self.X)
        
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Khởi tạo Dataset
train_dataset = MoonsDataset(X_train, y_train)
test_dataset = MoonsDataset(X_test, y_test)

# Khởi tạo DataLoader để quản lý batching và shuffle dữ liệu
batch_size = 32
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

print(f"Kích thước tập Train: {len(train_dataset)} mẫu")
print(f"Kích thước tập Test: {len(test_dataset)} mẫu")

### Trực quan hóa tập dữ liệu thô
Hãy vẽ đồ thị phân tán (Scatter Plot) của tập dữ liệu train để xem cấu trúc phân bổ phi tuyến của hai lớp.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(X_train[y_train.ravel() == 0, 0], X_train[y_train.ravel() == 0, 1], label='Lớp 0 (Đỏ)', color='red', alpha=0.7, edgecolors='k')
plt.scatter(X_train[y_train.ravel() == 1, 0], X_train[y_train.ravel() == 1, 1], label='Lớp 1 (Xanh)', color='blue', alpha=0.7, edgecolors='k')
plt.title("Tập dữ liệu huấn luyện phi tuyến Moons")
plt.xlabel("Đặc trưng 1 (Feature 1)")
plt.ylabel("Đặc trưng 2 (Feature 2)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## PHẦN 2: XÂY DỰNG MẠNG MLP PHÂN LOẠI NHỊ PHÂN

Mô hình MLP của chúng ta gồm:
* **Tầng đầu vào**: 2 node (Feature 1 và Feature 2).
* **Tầng ẩn 1**: 16 nơ-ron + kích hoạt ReLU.
* **Tầng ẩn 2**: 8 nơ-ron + kích hoạt ReLU.
* **Tầng đầu ra**: 1 nơ-ron + kích hoạt Sigmoid (cho ra xác suất từ 0 đến 1 của Lớp 1).

### Sự khác biệt với bài toán đa lớp:
Vì đây là phân loại nhị phân (chỉ có 2 lớp), ta chỉ cần 1 nơ-ron đầu ra đại diện cho lớp tích cực (Lớp 1) và áp dụng hàm kích hoạt **Sigmoid**. 
Hàm mất mát phù hợp nhất là **Binary Cross-Entropy Loss** (`nn.BCELoss`).

In [ ]:
class BinaryClassifierMLP(nn.Module):
    def __init__(self, input_dim=2, hidden_dim1=16, hidden_dim2=8, output_dim=1):
        super(BinaryClassifierMLP, self).__init__()
        
        # Định nghĩa các tầng liên kết đầy đủ
        self.fc1 = nn.Linear(input_dim, hidden_dim1)
        self.fc2 = nn.Linear(hidden_dim1, hidden_dim2)
        self.fc3 = nn.Linear(hidden_dim2, output_dim)
        
        # Các hàm kích hoạt
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        
        out = self.fc2(out)
        out = self.relu(out)
        
        # Đầu ra đi qua hàm Sigmoid để ép giá trị về khoảng [0, 1]
        out = self.sigmoid(self.fc3(out))
        return out

# Khởi tạo mô hình
model = BinaryClassifierMLP().to(device)
print(model)

# Định nghĩa BCE Loss và bộ tối ưu hóa Adam
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

---
## PHẦN 3: HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH

Chúng ta sẽ chạy vòng lặp huấn luyện trong 150 epochs và tính toán Accuracy.

In [ ]:
epochs = 150
loss_history = []
acc_history = []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for features, targets in train_loader:
        features, targets = features.to(device), targets.to(device)
        
        # 1. Clear gradients
        optimizer.zero_grad()
        
        # 2. Forward pass
        predictions = model(features)
        
        # 3. Calculate Loss
        loss = criterion(predictions, targets)
        
        # 4. Backward pass
        loss.backward()
        
        # 5. Update weights
        optimizer.step()
        
        # Thống kê
        running_loss += loss.item() * features.size(0)
        
        # Phân loại nhị phân: Ngưỡng xác suất >= 0.5 là nhãn 1, ngược lại nhãn 0
        preds_bin = (predictions >= 0.5).float()
        total += targets.size(0)
        correct += (preds_bin == targets).sum().item()
        
    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct / total
    loss_history.append(epoch_loss)
    acc_history.append(epoch_acc)
    
    if (epoch + 1) % 15 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2%}")

# Đánh giá trên tập test
model.eval()
test_correct = 0
test_total = 0
with torch.no_grad():
    for features, targets in test_loader:
        features, targets = features.to(device), targets.to(device)
        outputs = model(features)
        preds_bin = (outputs >= 0.5).float()
        test_total += targets.size(0)
        test_correct += (preds_bin == targets).sum().item()

print(f"\nĐộ chính xác cuối cùng trên tập Test: {test_correct / test_total:.2%}")

---
## PHẦN 4: TRỰC QUAN HÓA RANH GIỚI QUYẾT ĐỊNH (DECISION BOUNDARY)

Ranh giới quyết định (Decision Boundary) là đường phân tách giữa hai vùng dự đoán nhãn 0 và nhãn 1 của mô hình. 
Đối với các mô hình tuyến tính (như Hồi quy Logistic), ranh giới này luôn là một đường thẳng. Tuy nhiên, nhờ các tầng ẩn phi tuyến, mạng MLP có thể học được các ranh giới cong uốn lượn phức tạp để ôm khít dữ liệu.

In [ ]:
# 1. Vẽ đồ thị lịch sử huấn luyện
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(loss_history, color='red', label='Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Đồ thị Loss')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(acc_history, color='blue', label='Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Đồ thị Accuracy')
plt.grid(True, alpha=0.3)
plt.show()

# 2. Vẽ Decision Boundary
# Tạo một lưới tọa độ mịn (grid) phủ toàn bộ không gian dữ liệu kiểm thử
x_min, x_max = X_test[:, 0].min() - 0.5, X_test[:, 0].max() + 0.5
y_min, y_max = X_test[:, 1].min() - 0.5, X_test[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
grid_points = np.c_[xx.ravel(), yy.ravel()]

# Chuyển grid sang PyTorch Tensor để đưa qua mạng dự đoán xác suất
grid_tensor = torch.tensor(grid_points, dtype=torch.float32).to(device)

model.eval()
with torch.no_grad():
    probs = model(grid_tensor).cpu().numpy().reshape(xx.shape)

# Vẽ đồ thị ranh giới
plt.figure(figsize=(8, 5))
# Vẽ nền màu biểu thị xác suất dự đoán (vùng đỏ là Lớp 0, vùng xanh là Lớp 1)
plt.contourf(xx, yy, probs, levels=50, cmap='RdYlBu', alpha=0.6)

# Vẽ các điểm dữ liệu kiểm thử thực tế chồng lên trên
scatter = plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='RdYlBu', edgecolors='black', s=45)
plt.title(f"Ranh giới Quyết định của mạng MLP trên tập Test (Acc: {test_correct / test_total:.2%})")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.colorbar(scatter, label="Xác suất dự đoán thuộc Lớp 1")
plt.grid(True, alpha=0.3)
plt.show()